# POS Tagging Model Analysis

This notebook loads a trained POS tagging model, displays training curves, and tests on sample data.

In [2]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import json
import pickle

from model import create_model
from data import load_data_and_dataloaders
from evaluate import POSEvaluator
import config

## 1. Configuration and Setup

In [3]:
load_dotenv()
DATA_PATH = os.getenv("UD_DATA_PATH")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

language = 'en'
model_path = os.path.join(config.MODELS_DIR, f"best_model_{language}.pt")

print(f"Device: {device}")
print(f"Model path: {model_path}")
print(f"Data path: {DATA_PATH}")

Device: cuda
Model path: models/best_model_en.pt
Data path: /home/bagga/Desktop/pos_ud_project/data/raw/allzip/ud-treebanks-v2.17/


## 2. Load Data

In [4]:
train_loader, dev_loader, test_loader, word2idx, tag2idx, char2idx = load_data_and_dataloaders(
    DATA_PATH, language=language, batch_size=config.BATCH_SIZE, use_chars=True
)

idx2tag = {v: k for k, v in tag2idx.items()}
idx2word = {v: k for k, v in word2idx.items()}

print(f"Vocabulary size: {len(word2idx)}")
print(f"Number of tags: {len(tag2idx)}")
print(f"Tags: {list(tag2idx.keys())}")

Loading English dataset...
  Train sentences: 12544
  Dev sentences: 2001
  Test sentences: 2077
Building vocabularies...
  Vocabulary size: 8867
  POS tags: 18
  Character vocabulary: 83
Vocabulary size: 8867
Number of tags: 18
Tags: ['<PAD>', 'ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']


## 3. Load Trained Model

In [7]:
vocab_size = len(word2idx)
char_vocab_size = len(char2idx) if char2idx else 100
num_tags = len(tag2idx)

model = create_model(vocab_size, char_vocab_size, num_tags, device)

if os.path.exists(model_path):
    model.load_state_dict(torch.load('../'+model_path, map_location=device))
    print(f"Model loaded successfully from {model_path}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
else:
    print(f"Model file not found at {model_path}")

model.eval()

Model file not found at models/best_model_en.pt


SOTABiLSTMPOSTagger(
  (char_cnn): CharCNN(
    (char_embedding): Embedding(83, 50, padding_idx=0)
    (conv_layers): ModuleList(
      (0): Conv1d(50, 25, kernel_size=(3,), stride=(1,), padding=(2,))
      (1): Conv1d(50, 25, kernel_size=(4,), stride=(1,), padding=(3,))
      (2): Conv1d(50, 25, kernel_size=(5,), stride=(1,), padding=(4,))
    )
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (word_embedding): Embedding(8867, 256, padding_idx=0)
  (embedding_norm): LayerNorm((331,), eps=1e-05, elementwise_affine=True)
  (lstm): LSTM(331, 256, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (attention): MultiHeadAttention(
    (query): Linear(in_features=512, out_features=512, bias=True)
    (key): Linear(in_features=512, out_features=512, bias=True)
    (value): Linear(in_features=512, out_features=512, bias=True)
    (fc_out): Linear(in_features=512, out_features=512, bias=True)
    (dropout): Dropout(p=0.4, inplace=False)
  )
  (output_layer): Linear(in_feature

## 4. Training Curves (if available)

Note: You'll need to save training metrics during training to display curves.
This section assumes you have a training_history.json file.

In [ ]:
history_path = os.path.join(config.MODELS_DIR, f"training_history_{language}.json")

if os.path.exists(history_path):
    with open(history_path, 'r') as f:
        history = json.load(f)
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    axes[0, 0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    axes[0, 1].plot(history['dev_f1'], label='Dev F1', marker='o', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].set_title('Development F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    axes[1, 0].plot(history['dev_accuracy'], label='Dev Accuracy', marker='o', color='blue')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].set_title('Development Accuracy')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    axes[1, 1].plot(history['dev_precision'], label='Precision', marker='o', color='orange')
    axes[1, 1].plot(history['dev_recall'], label='Recall', marker='o', color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Score')
    axes[1, 1].set_title('Development Precision & Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Training history not found at {history_path}")
    print("To save training history, modify your training script to save metrics.")

## 5. Evaluate on Test Set

In [ ]:
evaluator = POSEvaluator(idx2tag)
test_metrics = evaluator.evaluate(model, test_loader, device, use_crf=config.USE_CRF)

print("="*60)
print("TEST SET RESULTS")
print("="*60)
print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1 Score:  {test_metrics['f1']:.4f}")
print()
print("Per-tag F1 scores:")
for tag, f1 in sorted(test_metrics['per_tag_f1'].items(), key=lambda x: x[1], reverse=True):
    if tag != '<PAD>':
        print(f"  {tag:12s}: {f1:.4f}")

## 6. Visualize Per-Tag Performance

In [ ]:
per_tag_f1 = {k: v for k, v in test_metrics['per_tag_f1'].items() if k != '<PAD>'}
tags = list(per_tag_f1.keys())
f1_scores = list(per_tag_f1.values())

plt.figure(figsize=(12, 6))
plt.bar(range(len(tags)), f1_scores, color='steelblue')
plt.xlabel('POS Tags')
plt.ylabel('F1 Score')
plt.title('Per-Tag F1 Scores on Test Set')
plt.xticks(range(len(tags)), tags, rotation=45, ha='right')
plt.ylim(0, 1.0)
plt.axhline(y=test_metrics['f1'], color='r', linestyle='--', label=f"Overall F1: {test_metrics['f1']:.4f}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Test on Custom Sentences

In [ ]:
def predict_sentence(model, sentence, word2idx, idx2tag, char2idx, device, use_crf=True):
    """
    Predict POS tags for a single sentence.
    """
    model.eval()
    
    words = sentence.split()
    word_ids = [word2idx.get(word.lower(), word2idx.get('<UNK>', 1)) for word in words]
    
    char_ids = []
    if char2idx:
        for word in words:
            char_ids.append([char2idx.get(c, char2idx.get('<UNK>', 1)) for c in word[:config.MAX_WORD_LENGTH]])
        max_word_len = max(len(cw) for cw in char_ids)
        char_ids = [cw + [0] * (max_word_len - len(cw)) for cw in char_ids]
    
    word_tensor = torch.tensor([word_ids], dtype=torch.long).to(device)
    char_tensor = torch.tensor([char_ids], dtype=torch.long).to(device) if char_ids else None
    mask = torch.ones_like(word_tensor, dtype=torch.bool).to(device)
    
    with torch.no_grad():
        if use_crf:
            predictions = model(word_tensor, char_tensor, mask)
            pred_tags = predictions[0]
        else:
            emissions = model(word_tensor, char_tensor, mask)
            pred_tags = emissions.argmax(dim=-1)[0].cpu().numpy()
    
    predicted_tags = [idx2tag[tag_id] for tag_id in pred_tags]
    
    return list(zip(words, predicted_tags))

In [ ]:
test_sentences = [
    "The quick brown fox jumps over the lazy dog",
    "She sells seashells by the seashore",
    "I am learning natural language processing",
    "The cat sat on the mat",
    "John visited Paris last summer"
]

print("="*60)
print("CUSTOM SENTENCE PREDICTIONS")
print("="*60)

for sentence in test_sentences:
    print(f"\nSentence: {sentence}")
    predictions = predict_sentence(model, sentence, word2idx, idx2tag, char2idx, device, use_crf=config.USE_CRF)
    print("Predictions:")
    for word, tag in predictions:
        print(f"  {word:15s} -> {tag}")
    print("-" * 60)

## 8. Interactive Prediction

In [ ]:
def interactive_prediction():
    """
    Interactive loop for testing custom sentences.
    """
    print("Enter a sentence to tag (or 'quit' to exit):")
    
    while True:
        sentence = input("\n> ")
        
        if sentence.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break
        
        if not sentence.strip():
            continue
        
        predictions = predict_sentence(model, sentence, word2idx, idx2tag, char2idx, device, use_crf=config.USE_CRF)
        
        print("\nPredictions:")
        for word, tag in predictions:
            print(f"  {word:15s} -> {tag}")

interactive_prediction()

## 9. Confusion Matrix (Optional)

Analyze which tags are commonly confused with each other.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

def get_all_predictions(model, data_loader, device, use_crf=True):
    """
    Get all predictions and true labels from a data loader.
    """
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in data_loader:
            words, chars, tags, mask = batch
            words = words.to(device)
            chars = chars.to(device) if chars is not None else None
            tags = tags.to(device)
            mask = mask.to(device)
            
            if use_crf:
                predictions = model(words, chars, mask)
                for pred, tag, m in zip(predictions, tags, mask):
                    valid_len = m.sum().item()
                    all_predictions.extend(pred[:valid_len])
                    all_labels.extend(tag[:valid_len].cpu().numpy())
            else:
                emissions = model(words, chars, mask)
                predictions = emissions.argmax(dim=-1)
                for pred, tag, m in zip(predictions, tags, mask):
                    valid_len = m.sum().item()
                    all_predictions.extend(pred[:valid_len].cpu().numpy())
                    all_labels.extend(tag[:valid_len].cpu().numpy())
    
    return all_predictions, all_labels

all_preds, all_labels = get_all_predictions(model, test_loader, device, use_crf=config.USE_CRF)

tags_for_cm = [tag for tag in tag2idx.keys() if tag != '<PAD>']
tag_indices = [tag2idx[tag] for tag in tags_for_cm]

cm = confusion_matrix(all_labels, all_preds, labels=tag_indices)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', 
            xticklabels=tags_for_cm, yticklabels=tags_for_cm)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - POS Tags')
plt.tight_layout()
plt.show()

print("Most confused tag pairs:")
confusion_pairs = []
for i in range(len(tags_for_cm)):
    for j in range(len(tags_for_cm)):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append((tags_for_cm[i], tags_for_cm[j], cm[i, j]))

confusion_pairs.sort(key=lambda x: x[2], reverse=True)
for true_tag, pred_tag, count in confusion_pairs[:10]:
    print(f"  {true_tag:12s} -> {pred_tag:12s}: {count:5d} times")

## 10. Save Analysis Results

In [ ]:
analysis_results = {
    'language': language,
    'model_path': model_path,
    'test_metrics': {
        'accuracy': float(test_metrics['accuracy']),
        'precision': float(test_metrics['precision']),
        'recall': float(test_metrics['recall']),
        'f1': float(test_metrics['f1']),
        'per_tag_f1': {k: float(v) for k, v in test_metrics['per_tag_f1'].items()}
    },
    'vocab_size': len(word2idx),
    'num_tags': len(tag2idx),
    'model_parameters': sum(p.numel() for p in model.parameters())
}

results_path = os.path.join(config.MODELS_DIR, f"analysis_results_{language}.json")
with open(results_path, 'w') as f:
    json.dump(analysis_results, f, indent=2)

print(f"Analysis results saved to {results_path}")